# Aula 03 · Vídeo 3 — Isolamento de Ferramentas: MCP, Sandboxing e Confidential Computing

> **Caso condutor:** Banco **Aurora** / **Aurora Assist** · **Tipo:** conceitual **+** prático
> **Doc 04:** seções 9-11 (confused deputy, MCP/isolamento, Confidential Computing)

## 🎯 Objetivos
- Entender **confused deputy**, **scope creep** e **prompt-as-payload** no *tool-calling*.
- Aplicar **capability tokens** de escopo mínimo e ver os abusos serem **bloqueados**.
- Reconhecer níveis de isolamento (**process/container/VM/capability**), *sandbox* (seccomp) e **TEE**.

> [!WARNING]
> ⚠️ **Uso ético.** As tentativas de abuso servem **apenas** para validar o isolamento em laboratório.

## 1. O risco do tool-calling (Doc 04, §9)

O **confused deputy** (Hardy, 1988) ocorre quando um agente com autoridade legítima é enganado a usá-la para o atacante — retomando a *Excessive Agency* (LLM08). Somam-se o **scope creep** (ferramentas acumulam permissões) e o **prompt-as-payload** (o parâmetro carrega o ataque). O exemplo assustador: se a ferramenta de saldo também **transferisse**, uma injeção viraria fraude. A defesa estrutural é **isolamento + menor privilégio** — que o MCP operacionaliza.

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Offline-first e determinístico (Restart & Run All sempre funciona).
os.environ.setdefault('PRESIDIO_MODE', 'deterministic')
os.environ.setdefault('TOKEN_VAULT_BACKEND', 'memory')

import random
random.seed(42)
print('ROOT =', ROOT)

ROOT = /Users/miyake-san/Projects/9mlet-llm-security/aula03_protecao_dados_isolamento


## 2. Catálogo de capacidades (policy-as-code · Doc 04, §10 / Snippet 3)

O **Snippet 3** expõe **apenas** a leitura de saldo. Aqui carregamos o catálogo declarativo (`app/policies/tool_policy.yaml`): note que **não existe** ferramenta `transferir` — a defesa contra o confused deputy é estrutural (a capacidade de escrita simplesmente não está no catálogo).

In [2]:
from src.security import ToolCatalog

catalog = ToolCatalog.from_yaml(ROOT / 'app' / 'policies' / 'tool_policy.yaml')
for name, cfg in catalog.to_dict().items():
    print(f"{name:28} read_only={cfg['read_only']}  scopes={cfg['scopes']}")
print('\ntransferir no catalogo?', 'transferir' in catalog.names())

consultar_saldo              read_only=True  scopes=['saldo:read']
consultar_horario_agencia    read_only=True  scopes=['agencia:read']

transferir no catalogo? False


## 3. Capability token de escopo mínimo (Doc 04, §10)

Cada chamada exige um **capability token** assinado (HMAC), de **vida curta** e **escopo mínimo**. O `ToolAuthorizer` verifica assinatura, expiração, ferramenta-alvo e escopo antes de autorizar.

In [3]:
from src.security import ToolAuthorizer, AuditTrail, issue_capability

authz = ToolAuthorizer(catalog=catalog, secret='lab-cap', audit=AuditTrail())
ok = issue_capability(subject='sessao-123', tool='consultar_saldo', scopes={'saldo:read'}, secret='lab-cap', ttl_seconds=120)
print('leitura de saldo autorizada?', authz.authorize(ok, 'consultar_saldo', 'saldo:read').to_dict())

leitura de saldo autorizada? {'allowed': True, 'tool': 'consultar_saldo', 'scope': 'saldo:read', 'reason': 'ok', 'output': None, 'latency_ms': 0.0}


## 4. Abusos bloqueados (confused deputy, scope creep, forjado, expirado)

Testamos as quatro tentativas clássicas — todas devem ser **negadas** com o motivo correto.

In [4]:
import dataclasses, time

demos = {}
# Confused deputy: token de saldo usado em OUTRA ferramenta.
demos['confused_deputy'] = authz.authorize(ok, 'consultar_horario_agencia', 'agencia:read')
# Scope creep: token sem o escopo pedido.
sem_escopo = issue_capability(subject='s', tool='consultar_saldo', scopes=set(), secret='lab-cap')
demos['scope_creep'] = authz.authorize(sem_escopo, 'consultar_saldo', 'saldo:read')
# Assinatura forjada.
forjado = dataclasses.replace(ok, signature='deadbeef')
demos['assinatura_forjada'] = authz.authorize(forjado, 'consultar_saldo', 'saldo:read')
# Token expirado.
expirado = issue_capability(subject='s', tool='consultar_saldo', scopes={'saldo:read'}, secret='lab-cap', ttl_seconds=-1)
demos['expirado'] = authz.authorize(expirado, 'consultar_saldo', 'saldo:read')

for nome, r in demos.items():
    print(f'{nome:20} allowed={r.allowed}  reason={r.reason}')

confused_deputy      allowed=False  reason=confused_deputy
scope_creep          allowed=False  reason=scope_creep
assinatura_forjada   allowed=False  reason=invalid_signature
expirado             allowed=False  reason=expired_token


## 5. Read-only estrutural (Doc 04, §9-10)

Mesmo que, por engano, uma ferramenta *read-only* expusesse um escopo de escrita, o autorizador o recusa. A operação de fraude não encontra caminho.

In [5]:
from src.security import ToolCatalog as TC
from src.security.schemas import ToolCapability

catalogo_erro = TC({'x': ToolCapability(name='x', scopes=frozenset({'x:write'}), read_only=True)})
authz_erro = ToolAuthorizer(catalog=catalogo_erro, secret='lab-cap')
tok = issue_capability(subject='s', tool='x', scopes={'x:write'}, secret='lab-cap')
print(authz_erro.authorize(tok, 'x', 'x:write').to_dict())

{'allowed': False, 'tool': 'x', 'scope': 'x:write', 'reason': 'write_denied_on_readonly_tool', 'output': None, 'latency_ms': 0.0}


## 6. Sandboxing com seccomp (Doc 04, §10)

No `docker-compose.yml`, o `mcp-server` roda *hardened*: non-root, `read_only`, `cap_drop:[ALL]`, **seccomp** (allowlist de *syscalls*) e em **rede interna sem egress**. O perfil está em `configs/seccomp.json` — `defaultAction` nega tudo que não estiver explicitamente permitido.

In [6]:
import json
sec = json.loads((ROOT / 'configs' / 'seccomp.json').read_text(encoding='utf-8'))
print('defaultAction     :', sec['defaultAction'])
print('syscalls no allowlist:', len(sec['syscalls'][0]['names']))

defaultAction     : SCMP_ACT_ERRNO
syscalls no allowlist: 142


## 7. Confidential Computing e TEEs (Doc 04, §11)

A criptografia protege o dado **em repouso** e **em trânsito**, mas durante o **processamento** ele fica em claro na RAM. O **Confidential Computing** fecha essa lacuna com **TEEs** (Intel **SGX**, AMD **SEV-SNP**, **Confidential GPUs**): o dado permanece cifrado **em uso**, inacessível até ao provedor de nuvem (Costan & Devadas, 2016). Para o Aurora, *workloads* ultrassensíveis podem rodar em **Confidential VMs/GPUs** — o teto do *defense-in-depth* aplicado ao dado. (Sem enclave real aqui: é a camada conceitual do vídeo.)

## 8. Trilha de auditoria como evidência (Doc 04, §12)

Cada decisão de autorização é registrada — insumo de *due diligence* (aprofundado na Aula 04).

In [7]:
for e in authz.audit.entries:
    print(f"{e.action:5} {e.tool:28} scope={e.scope} reason={e.reason}")

allow consultar_saldo              scope=saldo:read reason=ok
deny  consultar_horario_agencia    scope=agencia:read reason=confused_deputy
deny  consultar_saldo              scope=saldo:read reason=scope_creep
deny  consultar_saldo              scope=saldo:read reason=invalid_signature
deny  consultar_saldo              scope=saldo:read reason=expired_token


## ✅ Resumo e gancho

Isolamos ferramentas com **capability tokens** de escopo mínimo, bloqueamos *confused deputy*, *scope creep*, tokens forjados/expirados e escrita indevida, e revisamos *sandbox* (seccomp) e **TEE**. No **vídeo 4** integramos tudo: **Presidio + cofre de tokens + MCP isolado** no pipeline Zero Trust end-to-end e medimos as evidências de proteção.